# Zgodność anotacji - Wykop

Trzy osoby niezależnie oznaczyły te same 150 wpisów z Wykopu etykietami z promptu klasyfikacji
(`amc/classification/job_concern.py`):

- `ai` - czy wpis dotyczy sztucznej inteligencji (`yes`/`no`),
- `work` - czy wpis dotyczy pracy, kariery lub rynku pracy (`yes`/`no`),
- `stance` - co autor mówi o wpływie AI na pracę: `threat`, `reassurance`, `opportunity`,
  `mixed`, `none`.

Model w etapie 1 ocenia już tylko `ai` i `work`, więc z modelem porównujemy te dwie etykiety.
`stance` zostaje tylko jako zgodność między anotatorami (oznaczali je według starszych
wytycznych).

Pytania:

1. Na ile anotatorzy zgadzają się między sobą (czy etykiety są w ogóle dobrze zdefiniowane)?
2. Na ile model (Qwen przez Ollamę) zgadza się z anotatorami i czy mieści się w zakresie
   zgodności między ludźmi?

Pliki: `data/wykop/Anotacje_Wykop_<imię>.xlsx` (anotatorzy), etykiety modelu z
`data/wykop/wykop_stage1.csv` (tylko wiersze z aktualnego promptu).

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from amc.classification.job_concern import LABELS as MODEL_LABELS  # noqa: E402
from amc.classification.job_concern import PROMPT_HASH, YES_NO  # noqa: E402
from amc.classification.job_concern_details import STANCE  # noqa: E402

pd.set_option("display.max_colwidth", 200)
plt.rcParams["figure.figsize"] = (10, 4)

DATA = Path("../data/wykop")
OUT = DATA / "agreement"
OUT.mkdir(exist_ok=True)

ANNOTATORS = ["Mikołaj", "Patrycja", "Zuza"]
LABELS = ["ai", "work", "stance"]
CATEGORIES = {"ai": YES_NO, "work": YES_NO, "stance": STANCE}
SEED = 20261025
KEY = ["type", "entry_id"]

## Wczytanie danych

In [ ]:
def load_annotations(name):
    df = pd.read_excel(DATA / f"Anotacje_Wykop_{name}.xlsx", sheet_name="sample", dtype=str)
    df = df.dropna(subset=["entry_id"])
    for column in LABELS:
        df[column] = df[column].str.strip().str.lower()
    return df


sheets = {name: load_annotations(name) for name in ANNOTATORS}
base = sheets[ANNOTATORS[0]][[*KEY, "created_at", "url", "text"]].reset_index(drop=True)
for name, df in sheets.items():
    assert df[KEY].reset_index(drop=True).equals(base[KEY]), f"{name}: inne wpisy lub kolejność"
print("wpisów:", len(base), "| anotatorów:", len(sheets))

model_all = pd.read_csv(DATA / "wykop_stage1.csv", dtype=str, encoding="utf-8-sig")
model_all = model_all[model_all["prompt_hash"] == PROMPT_HASH]
model = base[KEY].merge(model_all, on=KEY, how="left", validate="one_to_one")
assert model["ai"].notna().all(), "brak etykiet modelu dla części wpisów"
print("model:", model["model"].unique(), "| prompt:", PROMPT_HASH[:8])

In [ ]:
# Jedna tabela: wiersz = wpis, kolumny = etykieta i oceniający
RATERS = [*ANNOTATORS, "model"]
ratings = {"model": model}
ratings.update(sheets)

table = base.copy()
for rater in RATERS:
    df = ratings[rater].reset_index(drop=True)
    for column in MODEL_LABELS if rater == "model" else LABELS:
        table[(column, rater)] = df[column].values


def raters_for(label):
    """Model ocenia tylko etykiety etapu 1."""
    return RATERS if label in MODEL_LABELS else ANNOTATORS


def labels_of(label, raters=ANNOTATORS):
    return np.column_stack([table[(label, rater)] for rater in raters])


# Wartości spoza słownika i etykiety niespójne z regułą starszych wytycznych
for rater in RATERS:
    for label in LABELS:
        if rater not in raters_for(label):
            continue
        bad = ~pd.Series(labels_of(label, [rater])[:, 0]).isin(CATEGORIES[label])
        if bad.any():
            print(f"{rater}: {bad.sum()} niepoprawnych wartości w {label}")
for rater in ANNOTATORS:
    rule = (
        (table[("stance", rater)] != "none")
        & ((table[("ai", rater)] == "no") | (table[("work", rater)] == "no"))
    )
    print(f"{rater:9} stance bez ai=yes i work=yes: {rule.sum()}")

## Rozkład etykiet

Duża przewaga jednej kategorii (np. `stance = none`) obniża kappę nawet przy wysokiej
zgodności procentowej (paradoks kappy), dlatego obok kapp podajemy też procent zgodności.

In [ ]:
distribution = pd.concat(
    {
        label: pd.DataFrame(
            {rater: table[(label, rater)].value_counts() for rater in raters_for(label)}
        ).reindex(CATEGORIES[label]).fillna(0).astype(int)
        for label in LABELS
    }
).astype("Int64")  # model nie ma stance: puste pola
distribution

## Miary zgodności

Trzy miary dla danych nominalnych, wyliczone wprost ze wzorów:

- **zgodność procentowa** - odsetek par ocen, które są takie same;
- **kappa Cohena** - dla pary oceniających (także model z człowiekiem), z poprawką na
  zgodność przypadkową wynikającą z rozkładu etykiet każdego z nich;
- **kappa Fleissa** - jedna liczba dla całej trójki anotatorów.

Przedziały ufności 95% z bootstrapu po wpisach (2000 losowań). Opis słowny kapp według
Landisa i Kocha (1977).

In [ ]:
def category_counts(values, categories):
    """Macierz wpis x kategoria: ilu oceniających wybrało daną kategorię."""
    return np.stack([(values == c).sum(axis=1) for c in categories], axis=1)


def percent_agreement(values):
    """Odsetek zgodnych par ocen; dla dwóch oceniających to odsetek identycznych ocen."""
    n = category_counts(values, np.unique(values))
    m = values.shape[1]
    return (((n**2).sum(axis=1) - m) / (m * (m - 1))).mean()


def cohen_kappa(values):
    a, b = values[:, 0], values[:, 1]
    p_o = np.mean(a == b)
    p_e = sum(np.mean(a == c) * np.mean(b == c) for c in np.union1d(a, b))
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else np.nan


def fleiss_kappa(values):
    n = category_counts(values, np.unique(values))
    p_e = ((n.sum(axis=0) / n.sum()) ** 2).sum()
    p_o = percent_agreement(values)
    return (p_o - p_e) / (1 - p_e) if p_e < 1 else np.nan


def bootstrap(statistic, values, iterations=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    draws = [
        statistic(values[rng.integers(0, len(values), len(values))])
        for _ in range(iterations)
    ]
    return np.nanpercentile(draws, [2.5, 97.5])


def landis_koch(value):
    """Opis słowny według Landisa i Kocha (1977)."""
    if np.isnan(value):
        return ""
    for limit, name in [(0, "brak"), (0.2, "niewielka"), (0.4, "dostateczna"),
                        (0.6, "umiarkowana"), (0.8, "znaczna")]:
        if value <= limit:
            return name
    return "prawie pełna"

In [ ]:
# Kontrola wzorów: pełna zgodność daje 1, a oceny losowe i niezależne około 0
check = np.array([["yes", "yes"], ["no", "no"], ["yes", "yes"], ["no", "no"]])
assert np.isclose(cohen_kappa(check), 1) and np.isclose(fleiss_kappa(check), 1)
rng = np.random.default_rng(1)
noise = rng.choice(["yes", "no"], size=(20000, 2))
assert abs(cohen_kappa(noise)) < 0.03 and abs(fleiss_kappa(noise)) < 0.03
assert np.isclose(percent_agreement(check), 1)

## 1. Zgodność między anotatorami

In [ ]:
def agreement_table(raters, labels=LABELS):
    rows = []
    for label in labels:
        values = labels_of(label, raters)
        low, high = bootstrap(fleiss_kappa, values)
        rows.append({
            "etykieta": label,
            "zgodność %": percent_agreement(values),
            "kappa Fleissa": fleiss_kappa(values),
            "95% CI": f"[{low:.2f}, {high:.2f}]",
            "ocena": landis_koch(fleiss_kappa(values)),
        })
    return pd.DataFrame(rows).set_index("etykieta")


humans = agreement_table(ANNOTATORS)
humans.to_csv(OUT / "agreement_annotators.csv", encoding="utf-8-sig")
humans.round(3)

In [ ]:
# Kappa Cohena dla każdej pary oceniających (model też jako oceniający)
pairs = []
for label in LABELS:
    for a, b in combinations(raters_for(label), 2):
        values = labels_of(label, [a, b])
        low, high = bootstrap(cohen_kappa, values)
        pairs.append({
            "etykieta": label,
            "para": f"{a} - {b}",
            "z modelem": "model" in (a, b),
            "zgodność %": np.mean(values[:, 0] == values[:, 1]),
            "kappa Cohena": cohen_kappa(values),
            "95% CI": f"[{low:.2f}, {high:.2f}]",
        })
pairs = pd.DataFrame(pairs)
pairs.to_csv(OUT / "cohen_pairs.csv", index=False, encoding="utf-8-sig")
pairs.pivot(index="para", columns="etykieta", values="kappa Cohena").round(3)

## 2. Model a anotatorzy

Dwa porównania:

- **model vs każdy anotator** (kappa Cohena) zestawione z kappami par ludzi - jeśli
  model zgadza się z ludźmi tak jak ludzie między sobą, mieści się w ich zakresie;
- **model vs etykieta złota**, czyli głos większości trojga anotatorów, dla `ai` i `work`.

In [ ]:
summary = (
    pairs.assign(kto=np.where(pairs["z modelem"], "model - człowiek", "człowiek - człowiek"))
    .groupby(["etykieta", "kto"])["kappa Cohena"]
    .agg(["mean", "min", "max"])
    .unstack("kto")
)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, label in zip(axes, LABELS):
    part = pairs[pairs["etykieta"] == label]
    colors = np.where(part["z modelem"], "tab:orange", "tab:blue")
    ax.barh(part["para"], part["kappa Cohena"], color=colors)
    ax.axvline(0.6, color="grey", ls=":", lw=1)
    ax.set_title(label)
    ax.set_xlim(min(0, part["kappa Cohena"].min() - 0.05), 1)
axes[0].set_ylabel("para oceniających")
fig.suptitle("Kappa Cohena: niebieskie - ludzie, pomarańczowe - model (linia: 0,6)")
plt.tight_layout()
plt.savefig(OUT / "cohen_pairs.png", dpi=120)
plt.show()

In [ ]:
def majority(values):
    """Głos większości w wierszu; None przy remisie."""
    result = []
    for row in values:
        labels, counts = np.unique(row, return_counts=True)
        top = counts.max()
        result.append(labels[counts.argmax()] if (counts == top).sum() == 1 else None)
    return np.array(result, dtype=object)


for label in LABELS:
    table[(label, "gold")] = majority(labels_of(label))
    votes = labels_of(label)
    table[(label, "jednomyślnie")] = (votes == votes[:, [0]]).all(axis=1)

print("remisy w stance:", table[("stance", "gold")].isna().sum())
print("jednomyślne oceny:", {
    label: int(table[(label, "jednomyślnie")].sum()) for label in LABELS
})

In [ ]:
def per_class(truth, predicted, categories):
    rows = []
    for c in categories:
        tp = np.sum((predicted == c) & (truth == c))
        fp = np.sum((predicted == c) & (truth != c))
        fn = np.sum((predicted != c) & (truth == c))
        precision = tp / (tp + fp) if tp + fp else np.nan
        recall = tp / (tp + fn) if tp + fn else np.nan
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan
        rows.append({"kategoria": c, "złota": tp + fn, "model": tp + fp,
                     "precyzja": precision, "czułość": recall, "F1": f1})
    return pd.DataFrame(rows).set_index("kategoria")


vs_gold = []
for label in MODEL_LABELS:
    keep = table[(label, "gold")].notna().to_numpy()
    values = np.column_stack([table[(label, "gold")], table[(label, "model")]])[keep]
    low, high = bootstrap(cohen_kappa, values)
    vs_gold.append({
        "etykieta": label,
        "wpisów": int(keep.sum()),
        "trafność": np.mean(values[:, 0] == values[:, 1]),
        "kappa Cohena": cohen_kappa(values),
        "95% CI": f"[{low:.2f}, {high:.2f}]",
        "ocena": landis_koch(cohen_kappa(values)),
        "kappa ludzi (średnia par)": summary.loc[label, ("mean", "człowiek - człowiek")],
    })
vs_gold = pd.DataFrame(vs_gold).set_index("etykieta")
vs_gold.to_csv(OUT / "model_vs_gold.csv", encoding="utf-8-sig")
vs_gold.round(3)

In [ ]:
# Czy dołączenie modelu jako czwartego oceniającego psuje zgodność grupy?
compare = pd.DataFrame({
    "Fleiss: 3 anotatorów": humans.loc[list(MODEL_LABELS), "kappa Fleissa"],
    "Fleiss: 3 anotatorów + model": agreement_table(RATERS, MODEL_LABELS)["kappa Fleissa"],
})
compare.round(3)

### Macierze pomyłek i miary per kategoria (model vs złota)

In [ ]:
for label in MODEL_LABELS:
    keep = table[(label, "gold")].notna()
    truth = table[(label, "gold")][keep].astype(str).to_numpy()
    predicted = table[(label, "model")][keep].to_numpy()
    print(f"\n=== {label} (wiersze: złota, kolumny: model)")
    display(pd.crosstab(
        pd.Categorical(truth, CATEGORIES[label]), pd.Categorical(predicted, CATEGORIES[label]),
        rownames=["złota"], colnames=["model"], dropna=False,
    ))
    scores = per_class(truth, predicted, CATEGORIES[label])
    scores.to_csv(OUT / f"per_class_{label}.csv", encoding="utf-8-sig")
    display(scores.round(3))

## 3. Próba a cały zbiór

Jeśli próba do anotacji była losowana warstwowo (np. z nadreprezentacją wpisów o AI),
trafność i czułość z próby nie przenoszą się wprost na cały zbiór. Poniżej rozkład par
etykiet modelu (`ai`, `work`) w próbie i we wszystkich wpisach sklasyfikowanych aktualnym
promptem.

In [ ]:
def pairs_of(ai, work):
    return ("ai=" + pd.Series(ai).astype(str).values + ", work="
            + pd.Series(work).astype(str).values)


population = pd.DataFrame({
    "próba": pd.Series(pairs_of(table[("ai", "model")], table[("work", "model")]))
    .value_counts(normalize=True),
    "cały zbiór": pd.Series(pairs_of(model_all["ai"], model_all["work"]))
    .value_counts(normalize=True),
}).fillna(0)
print("wpisów sklasyfikowanych aktualnym promptem:", len(model_all))
population.round(3)

## 4. Rozbieżności do omówienia

Wpisy, w których `ai` lub `work` modelu różni się od większości anotatorów, oraz wpisy,
przy których anotatorzy różnili się w `stance`. Te listy są podstawą do doprecyzowania
wytycznych i promptu. Zapisujemy je do `data/wykop/agreement/`.

In [ ]:
def show(mask, labels):
    columns = {}
    for label in labels:
        for rater in [*raters_for(label), "gold"]:
            columns[f"{label}: {rater}"] = table[(label, rater)]
    view = pd.DataFrame({
        "url": table["url"],
        "text": table["text"].str.replace("\n", " ").str.slice(0, 300),
        **columns,
    })
    return view[mask.to_numpy()]


model_split = pd.Series(False, index=table.index)
for label in MODEL_LABELS:
    model_split |= table[(label, "model")] != table[(label, "gold")]
human_split = ~table[("stance", "jednomyślnie")]
print("model inaczej niż większość (ai lub work):", model_split.sum(),
      "| anotatorzy niezgodni w stance:", human_split.sum())

show(model_split, MODEL_LABELS).to_csv(OUT / "model_vs_gold_stage1.csv", encoding="utf-8-sig")
show(human_split, ["stance"]).to_csv(OUT / "annotators_split_stance.csv", encoding="utf-8-sig")
show(model_split, MODEL_LABELS)

In [ ]:
show(human_split, ["stance"])

## Jak czytać wyniki

- **Ludzie między sobą.** Kappa Fleissa powyżej 0,8 to zgodność prawie pełna, 0,6-0,8
  znaczna (Landis i Koch 1977). Jeśli zgodność jest niska, problemem jest definicja
  kategorii, nie model - najpierw doprecyzujcie wytyczne na podstawie listy rozbieżności.
- **Model.** Jeśli kappa model-człowiek jest w zakresie kapp człowiek-człowiek, model może
  zastąpić anotatora. Jeśli jest wyraźnie niższa, trzeba poprawić prompt albo model.
- Kappa przy rzadkich kategoriach bywa niska mimo wysokiej zgodności procentowej - wtedy
  patrz też na liczby w macierzy pomyłek.